In [ ]:
import psycopg2
from datetime import datetime

# ── CONFIGURACIÓN DE CONEXIONES ──────────────────────────────────────────
# Base de datos origen (Donde tienes la tabla usuarios con 'country', 'birthdate', etc.)
DB_ORIGEN = {
    "dbname": "tu_db_origen",
    "user": "postgres",
    "password": "tu_password",
    "host": "localhost",
    "port": "5432"
}

# Base de datos destino (Tu nuevo almacén de datos OLAP)
DB_DESTINO = {
    "dbname": "tu_db_olap",
    "user": "postgres",
    "password": "tu_password",
    "host": "localhost",
    "port": "5432"
}

# Diccionario auxiliar para deducir el continente automáticamente
DICCIONARIO_CONTINENTES = {
    "méxico": "LATAM",
    "panamá": "LATAM",
    "españa": "Europa",
    "colombia": "LATAM",
    "ee.uu.": "Norteamérica",
    "united states": "Norteamérica"
}

def calcular_edad_y_rango(fecha_nacimiento):
    if not fecha_nacimiento:
        return 0, "Desconocido"
    
    # Calcular edad exacta basándonos en el año actual (2026)
    hoy = datetime.now()
    edad = hoy.year - fecha_nacimiento.year - ((hoy.month, hoy.day) < (fecha_nacimiento.month, fecha_nacimiento.day))
    
    # Asignar segmento o rango de edad para marketing analítico
    if edad < 18:
        rango = "Menor de edad"
    elif 18 <= edad <= 24:
        rango = "18-24"
    elif 25 <= edad <= 34:
        rango = "25-34"
    else:
        rango = "35+"
        
    return edad, rango

def iniciar_migracion():
    try:
        # 1. Conectar a Destino para crear las tablas analíticas si no existen
        conn_dest = psycopg2.connect(**DB_DESTINO)
        cursor_dest = conn_dest.cursor()
        
        print("Creando tablas analíticas en la base de datos destino...")
        cursor_dest.execute("""
            CREATE TABLE IF NOT EXISTS dim_ubicacion (
                id_ubicacion SERIAL PRIMARY KEY,
                continente VARCHAR(50),
                pais VARCHAR(100),
                estado VARCHAR(100)
            );
            
            CREATE TABLE IF NOT EXISTS dim_usuario (
                id_usuario VARCHAR(100) PRIMARY KEY,
                nombre VARCHAR(150),
                edad INT,
                rango_edad VARCHAR(30),
                tipo_membresia VARCHAR(50)
            );
        """)
        conn_dest.commit()

        # 2. Conectar a la base de datos Origen para extraer los datos crudos
        conn_orig = psycopg2.connect(**DB_ORIGEN)
        cursor_orig = conn_orig.cursor()
        
        print("Extrayendo usuarios de la base de datos transaccional...")
        cursor_orig.execute("SELECT cognito_id, username, birthdate, country, state FROM usuarios;")
        usuarios_crudos = cursor_orig.fetchall()

        # 3. Procesar, Transformar y Cargar elemento por elemento
        print(f"Transformando e insertando {len(usuarios_crudos)} registros...")
        for row in usuarios_crudos:
            cognito_id, username, birthdate, country, state = row
            
            # Limpieza de strings y normalización para el diccionario
            pais_limpio = country.strip() if country else "Desconocido"
            estado_limpio = state.strip() if state else "Desconocido"
            continente = DICCIONARIO_CONTINENTES.get(pais_limpio.lower(), "Otros")
            
            # ── TRANSFORMACIÓN ANALÍTICA ──
            edad, rango_edad = calcular_edad_y_rango(birthdate)
            tipo_membresia = "Gratuito" # Valor por defecto inicial para tu startup
            
            # Insertar u obtener ubicación para evitar duplicar países/estados idénticos
            cursor_dest.execute("""
                INSERT INTO dim_ubicacion (continente, pais, estado)
                VALUES (%s, %s, %s)
                ON CONFLICT DO NOTHING;
            """, (continente, pais_limpio, estado_limpio))
            
            # Asignar la data mapeada a la tabla de usuarios OLAP
            cursor_dest.execute("""
                INSERT INTO dim_usuario (id_usuario, nombre, edad, rango_edad, tipo_membresia)
                VALUES (%s, %s, %s, %s, %s)
                ON CONFLICT (id_usuario) DO UPDATE 
                SET nombre = EXCLUDED.nombre, edad = EXCLUDED.edad, rango_edad = EXCLUDED.rango_edad;
            """, (cognito_id, username, edad, rango_edad, tipo_membresia))
            
        # Guardar cambios permanentemente en destino
        conn_dest.commit()
        
        # Cerrar flujos de comunicación de base de datos
        cursor_orig.close()
        conn_orig.close()
        cursor_dest.close()
        conn_dest.close()
        
        print("¡Proceso ETL completado exitosamente! Tu modelo OLAP está poblado.")

    except Exception as error:
        print(f"Error durante la migración de datos: {error}")

if __name__ == "__main__":
    iniciar_migracion()